In [1]:
import geopandas as gpd
from shapely.geometry import LineString, MultiLineString, Point
import numpy as np
import pandas as pd

# ----------------------------
# Step 0: Road hierarchy
# ----------------------------
road_hierarchy = {'A':4,'B':3,'C':2,'D':1}

# ----------------------------
# Step 1: Load data
# ----------------------------
buildings = gpd.read_file(r"F:\KMC_GIS_Road Survey\House Numbering Project\House Numbering Trial\Python\Complete_Numbering\Ward 14\Data\Building_14_Centroids.shp")
roads = gpd.read_file(r"F:\KMC_GIS_Road Survey\House Numbering Project\House Numbering Trial\Python\Complete_Numbering\Ward 14\Data\Road_14.shp")

buildings = buildings.to_crs(roads.crs)

roads['Unique_ID'] = roads['Unique_ID'].astype(str).str.strip()

# ----------------------------
# Step 2: explode multilines
# ----------------------------
roads = roads.explode(index_parts=False).reset_index(drop=True)

# ----------------------------
# Step 3: nearest road to building
# ----------------------------
def nearest_road(row):

    centroid = row.geometry
    dist = roads.geometry.distance(centroid)

    return roads.loc[dist.idxmin(),'Unique_ID']

buildings['Road_ID'] = buildings.apply(nearest_road,axis=1)

# ----------------------------
# Step 4: Find parent road
# ----------------------------
parent = {}
parent_dist = {}

for idx, road in roads.iterrows():

    geom = road.geometry

    if isinstance(geom,MultiLineString):
        geom = max(geom.geoms,key=lambda x:x.length)

    start_node = Point(geom.coords[0])

    level = road_hierarchy.get(road['road_class'],0)

    best_parent = None
    best_level = -1
    best_dist = None

    for j, other in roads.iterrows():

        if road['Unique_ID']==other['Unique_ID']:
            continue

        other_geom = other.geometry

        if isinstance(other_geom,MultiLineString):
            other_geom = max(other_geom.geoms,key=lambda x:x.length)

        if other_geom.distance(start_node) < 0.5:

            other_level = road_hierarchy.get(other['road_class'],0)

            if other_level >= level and other_level > best_level:

                best_level = other_level
                best_parent = other['Unique_ID']
                best_dist = other_geom.project(start_node)

    parent[road['Unique_ID']] = best_parent
    parent_dist[road['Unique_ID']] = best_dist

# ----------------------------
# Step 5: Compute x and y
# ----------------------------
x_dict = {}
y_dict = {}

for rid in roads['Unique_ID']:

    p = parent.get(rid)

    if p is None:
        x_dict[rid] = None
        y_dict[rid] = None
        continue

    y_dict[rid] = parent_dist.get(rid)

    pp = parent.get(p)

    if pp is None:
        x_dict[rid] = None
    else:
        x_dict[rid] = parent_dist.get(p)

        
# ----------------------------
# Step 6: Distance along C road (z)
# ----------------------------
def distance_along_road(row):

    centroid = row.geometry
    road_match = roads[roads['Unique_ID']==row['Road_ID']]

    if road_match.empty:
        return np.nan

    road_geom = road_match.geometry.values[0]

    if isinstance(road_geom,MultiLineString):
        road_geom = max(road_geom.geoms,key=lambda x:x.length)

    nearest = road_geom.interpolate(road_geom.project(centroid))
    dist = road_geom.project(nearest)

    return int(dist)

buildings['z'] = buildings.apply(distance_along_road,axis=1)

# ----------------------------
# Step 7: Left / Right detection
# ----------------------------
def left_right(row):

    centroid = row.geometry

    road_match = roads[roads['Unique_ID']==row['Road_ID']]

    if road_match.empty:
        return None

    road_geom = road_match.geometry.values[0]

    if isinstance(road_geom,MultiLineString):
        road_geom = max(road_geom.geoms,key=lambda x:x.length)

    d = road_geom.project(centroid)

    delta = 0.5

    p1 = road_geom.interpolate(max(0,d-delta))
    p2 = road_geom.interpolate(min(road_geom.length,d+delta))

    x1,y1 = p1.x,p1.y
    x2,y2 = p2.x,p2.y

    px,py = centroid.x,centroid.y

    cross = (x2-x1)*(py-y1)-(y2-y1)*(px-x1)

    return 'Left' if cross>0 else 'Right'

buildings['Side'] = buildings.apply(left_right,axis=1)

# ----------------------------
# Step 8: Assign house number
# ----------------------------
def assign_number(row):

    z = row['z']
    side = row['Side']

    if pd.isna(z) or side is None:
        return np.nan

    number = int(z)

    # Odd Even rule (same as before)
    if side=='Left':
        if number%2==0:
            number+=1
    else:
        if number%2==1:
            number+=1

    y = y_dict.get(row['Road_ID'])
    x = x_dict.get(row['Road_ID'])

    if x and y:
        return f"{int(x)}/{int(y)}-{number}"
    elif y:
        return f"{int(y)}-{number}"
    else:
        return f"{number}"

buildings['Final_House_Number'] = buildings.apply(assign_number,axis=1)
print(buildings.head(20))


# ----------------------------
# Assign Road_Con_Name based on house number structure
# ----------------------------
def assign_road_con(row):

    hn = str(row['Final_House_Number'])

    if pd.isna(row['Final_House_Number']):
        return None

    rid = row['Road_ID']

    # Case 1: x/y-z → take parent of parent
    if '/' in hn:
        p = parent.get(rid)
        pp = parent.get(p) if p else None
        return pp if pp else p if p else rid

    # Case 2: y-z → take parent
    elif '-' in hn:
        p = parent.get(rid)
        return p if p else rid

    # Case 3: z only → take itself
    else:
        return rid

buildings['Road_Con_Name'] = buildings.apply(assign_road_con, axis=1)

# # ----------------------------
# # Step 9: Save buildings
# # ----------------------------
# output_building = r"F:\KMC_GIS_Road Survey\House Numbering Project\House Numbering Trial\Python\Complete_Numbering\Ward 14\Output\Corrected\Ward14_Final.shp"

# buildings.to_file(output_building)

# print("Building numbering saved")

# # ----------------------------
# # Step 10: Create link lines
# # ----------------------------
# links = []

# for idx,row in buildings.iterrows():

#     pt = row.geometry

#     road_geom = roads.loc[roads['Unique_ID']==row['Road_ID'],'geometry'].values[0]

#     if isinstance(road_geom,MultiLineString):
#         road_geom = max(road_geom.geoms,key=lambda x:x.length)

#     nearest = road_geom.interpolate(road_geom.project(pt))

#     line = LineString([(pt.x,pt.y),(nearest.x,nearest.y)])

#     links.append(line)

# links_gdf = gpd.GeoDataFrame(
#     buildings[['Road_ID','Final_House_Number','z','Side']],
#     geometry=links,
#     crs=buildings.crs
# )

# # ----------------------------
# # Step 11: Save links
# # ----------------------------
# output_links = r"F:\KMC_GIS_Road Survey\House Numbering Project\House Numbering Trial\Python\Complete_Numbering\Ward 14\Output\Corrected\Ward14_FinalLink.shp"

# links_gdf.to_file(output_links)

# print("Link shapefile saved")

C:\Users\arunb\AppData\Roaming\Python\Python39\site-packages\pyogrio\raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D LineString' is converted to 'LineString Z'
  return ogr_read(
C:\Users\arunb\AppData\Roaming\Python\Python39\site-packages\shapely\measurement.py:72: RuntimeWarning: invalid value encountered in distance
  return lib.distance(a, b, **kwargs)
C:\Users\arunb\AppData\Roaming\Python\Python39\site-packages\shapely\measurement.py:72: RuntimeWarning: invalid value encountered in distance
  return lib.distance(a, b, **kwargs)
C:\Users\arunb\AppData\Roaming\Python\Python39\site-packages\shapely\measurement.py:72: RuntimeWarning: invalid value encountered in distance
  return lib.distance(a, b, **kwargs)
C:\Users\arunb\AppData\Roaming\Python\Python39\site-packages\shapely\measurement.py:72: RuntimeWarning: invalid value encountered in distance
  return lib.distance(a, b, **kwargs)
C:\Users\arunb\AppData\Roaming\Python\Python39\sit

    gid    id roof_type ward_no  perimeter  area build_type  \
0     1  None       TIN    None        NaN   NaN       None   
1     2  None       TIN    None        NaN   NaN       None   
2     3  None       TIN    None        NaN   NaN       None   
3     4  None       TIN    None        NaN   NaN       None   
4     5  None       TIN    None        NaN   NaN       None   
5     6  None       TIN    None        NaN   NaN       None   
6     7  None       TIN    None        NaN   NaN       None   
7     8  None  Concrete    None        NaN   NaN       None   
8     9  None       TIN    None        NaN   NaN       None   
9    10  None       TIN    None        NaN   NaN       None   
10   11  None       TIN    None        NaN   NaN       None   
11   12  None  Concrete    None        NaN   NaN       None   
12   13  None       TIN    None        NaN   NaN       None   
13   14  None       TIN    None        NaN   NaN       None   
14   15  None       TIN    None        NaN   NaN       

<ipython-input-1-e6ac441a2e90>:234: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  buildings.to_file(output_building)


Building numbering saved


C:\Users\arunb\AppData\Roaming\Python\Python39\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Final_House_Number' to 'Final_Hous'
  ogr_write(
C:\Users\arunb\AppData\Roaming\Python\Python39\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Road_Con_Name' to 'Road_Con_N'
  ogr_write(
C:\Users\arunb\AppData\Roaming\Python\Python39\site-packages\shapely\linear.py:88: RuntimeWarning: invalid value encountered in line_locate_point
  return lib.line_locate_point(line, other)


Link shapefile saved


<ipython-input-1-e6ac441a2e90>:269: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  links_gdf.to_file(output_links)
C:\Users\arunb\AppData\Roaming\Python\Python39\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Final_House_Number' to 'Final_Hous'
  ogr_write(
